# Occurrence Records of African Bats with Taxonomic, Geographic, and Temporal Annotations Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL for the dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.swcc-jqh9/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata as a Python object
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets and their fields, referencing them by their `@id`.

In [ ]:
# List available RecordSets
print("Available record sets (by @id):")
for rs in dataset.record_sets:
    print(f"  - @id: {rs['@id']}, name: {rs.get('name', '[no name]')}")

# As an example, display fields for each RecordSet
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
print("\nFields per RecordSet:")
for rs in dataset.record_sets:
    print(f"\nRecordSet @id: {rs['@id']} ({rs.get('name','[no name]')})")
    if 'field' in rs:
        for field in rs['field']:
            if isinstance(field, dict):
                print(f"    Field @id: {field.get('@id','?')} | name: {field.get('name','?')}")
            else:
                print(f"    Field @id: {field}")
    else:
        print("    [No fields listed]")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For demonstration, we'll use the first record set (if any)
print("\nLoading records from all record sets:")
dataframes = {}
loaded = False
for rs in dataset.record_sets:
    record_set_id = rs['@id']
    try:
        print(f"Loading records for RecordSet @id={record_set_id} ...")
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"  Loaded {len(df)} records. Columns: {list(df.columns)}")
            loaded = True
        else:
            print("  No records found.")
    except Exception as e:
        print(f"  Error loading records: {e}")

if not loaded:
    print("No records loaded. Check that record sets have records defined and accessible.")
else:
    # Show data from the first loaded dataframe
    first_rs = next(iter(dataframes))
    print(f"\nSample data from RecordSet @id={first_rs}:")
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

> **Note**: We'll select numeric and group fields by `@id` as found in the previous overview/output. If the dataset contains latitude/longitude or year fields (`latitude`, `longitude`, `year`), these will be demonstrated (please adapt to actual field `@id`s if different).

In [ ]:
# Example: Filter and analyze a numeric field in the first loaded DataFrame.

if dataframes:
    record_set_id = next(iter(dataframes))
    df = dataframes[record_set_id]
    print(f"Working with RecordSet @id: {record_set_id}")
    # Try to pick a numeric field by common name; fallback to first numeric
    numeric_candidates = [c for c in df.columns if any(n in c.lower() for n in ['latitude', 'longitude', 'lat', 'long', 'year', 'numeric', 'count'])]
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        print(f"Numeric field selected by @id: {numeric_field}")
        # Try to convert to numeric (ignore errors)
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        # Filter to values greater than a threshold (demonstration: 0)
        threshold = 0
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records where {numeric_field} > {threshold}: {len(filtered_df)} records")
        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        )
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Pick a group field (e.g., country, genus, family)
        group_candidates = [c for c in df.columns if any(g in c.lower() for g in ['country', 'family', 'genus', 'species'])]
        if group_candidates:
            group_field = group_candidates[0]
            print(f"\nGrouping by field @id: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(grouped_df.head())
        else:
            print("No group field found for demonstration.")
    else:
        print("No numeric field found for EDA demonstration.")
else:
    print("No dataframes loaded for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# If the data and fields are available, create a basic visualization
if dataframes:
    df = next(iter(dataframes.values()))
    num_fields = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    if num_fields:
        field = num_fields[0]
        plt.figure(figsize=(8,5))
        sns.histplot(df[field].dropna(), bins=30, kde=True)
        plt.title(f"Distribution of {field} (@id)")
        plt.xlabel(field)
        plt.ylabel("Count")
        plt.show()
    else:
        print("No numeric field found for visualization.")
else:
    print("No dataframes loaded for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we loaded the FAIR^2 "Occurrence Records of African Bats" dataset using the `mlcroissant` library. By referencing entities by their `@id`, we programmatically explored available record sets and their fields, loaded tables into pandas DataFrames, performed simple cleaning and aggregations, and visualized a selected numeric attribute.

The dataset provides rich biogeographic and taxonomic data suitable for biodiversity and ecological research. We encourage further analysis using specific field `@id`s and domain knowledge relevant to African chiropteran diversity.